In [2]:
import os, io, re, json, warnings
import numpy as np
import pandas as pd
import boto3

LOCAL_MODE = False

BUCKET  = "nyp-26s1-iti113"
TEAM_ID = "team14"
STUDENT_ID = "s1401"

COURSE = "ITI113"
SEMESTER = "26S1"


PROJECT_NAME01 = "airbnb-instant-booking"
PROJECT_NAME02 = "airbnb-listings"
PROJECT_NAME99 = "airbnb-recommendation"

PREFIX99  = f"iti113/{TEAM_ID}/data/{PROJECT_NAME99}"
PREFIX01  = f"iti113/{TEAM_ID}/data/{PROJECT_NAME01}"
PREFIX02  = f"iti113/{TEAM_ID}/data/{PROJECT_NAME02}"

# Initialize AWS Session & S3 Client
boto_session = boto3.Session()
region = boto_session.region_name or 'us-east-1'
s3 = boto_session.client('s3')

# Try loading SageMaker Session & Execution Role (fallback gracefully if using external IAM user/keys)
try:
    import sagemaker
    session = sagemaker.Session(boto_session=boto_session)
    role = sagemaker.get_execution_role()
except Exception:
    session = None
    role = "arn:aws:iam::default:role/LocalOrExternalRole"


# Separate output prefix for instant-booking classification
PREFIX = f"iti113/{TEAM_ID}/data/airbnb-instant-booking"
LOCAL_OUT = "processed_instant_booking"

# ---- data source ----
DATA_CANDIDATES = [
    "Listings.csv", 
    os.path.join("data", "Listings.csv"),
    "/mnt/user-data/uploads/Listings.csv"
]
DATA_PATH = next((p for p in DATA_CANDIDATES if os.path.exists(p)), DATA_CANDIDATES[0])

# ---- modelling constants (Binary Classification) ----
TARGET_COLUMN         = "instant_bookable"   # Binary target flag: 't'/'f' or 1/0
TARGET_POSITIVE_CLASS = 1                    # 1 = Instant booking enabled, 0 = Requires approval
RANDOM_STATE          = 42
TEST_SIZE             = 0.20
SNAPSHOT_DATE         = pd.Timestamp("2021-03-01")   # Data as-of date (max host_since = 2021-02-26)

# ---- classification & evaluation policy ----
DECISION_THRESHOLD    = 0.50                 # Default baseline; tune post-calibration
CALIBRATION_METHOD    = "isotonic"           # Ensures output probabilities are reliable for rankers
STRATIFY_COLUMNS      = ["city", "instant_bookable"]  # Joint stratification for regional/class balance

# ---- documented policy constants (outlier & feature engineering policies) ----
TOP_K_AMENITIES       = 30      # Amenity vocabulary size (capturing self-checkin/keypads), fit on train
COORD_DECIMALS        = 2       # Coordinate rounding: 2 dp ~ 1.1 km cells (geo-privacy)
PRICE_CAP_QUANTILE    = 0.995   # Feature-level price winsorisation per city, fit on train
COUNT_CAP_QUANTILE    = 0.99    # host_total_listings_count cap, fit on train
MIN_NIGHTS_CAP        = 365     # Domain constant: >1 year minimum stay is not nightly rental
MAX_NIGHTS_CAP        = 1125    # Platform default ceiling; larger values are sentinels
BEDROOMS_CAP          = 16      # Platform max party size; larger values are stability caps
SPARSITY_THRESHOLD    = 0.80    # Columns with >80% missing are dropped as unusable

# ---- fixed external reference points (NOT computed from data) ----
CITY_CENTERS = {                     # (lat, lon) of a canonical central landmark
    "Paris":          (48.8530,   2.3499),   # Notre-Dame
    "New York":       (40.7580, -73.9855),   # Times Square / Midtown
    "Sydney":         (-33.8568, 151.2153),  # Circular Quay
    "Rome":           (41.8986,  12.4769),   # Pantheon
    "Rio de Janeiro": (-22.9711, -43.1822),  # Copacabana
    "Istanbul":       (41.0054,  28.9768),   # Sultanahmet
    "Mexico City":    (19.4326, -99.1332),   # Zocalo
    "Bangkok":        (13.7460, 100.5340),   # Siam
    "Cape Town":      (-33.9221, 18.4231),   # City Centre
    "Hong Kong":      (22.2819, 114.1582),   # Central
}

# ---- verification output ----
print(f"Mode         : {'LOCAL (no SageMaker/S3)' if LOCAL_MODE else 'SageMaker / S3 Active'}")
print(f"Bucket       : {BUCKET}")
print(f"Prefix       : {PREFIX}")
print(f"Region       : {region}")
print(f"Role         : {role.split('/')[-1] if role else 'None'}")
print(f"Team ID      : {TEAM_ID}")
print(f"Student ID   : {STUDENT_ID}")
print(f"Task         : Binary Classification")
print(f"Target       : {TARGET_COLUMN}")
print(f"Data path    : {DATA_PATH}")
print(f"Snapshot     : {SNAPSHOT_DATE.date()}")
print(f"numpy {np.__version__} | pandas {pd.__version__}")

Mode         : SageMaker / S3 Active
Bucket       : nyp-26s1-iti113
Prefix       : iti113/team14/data/airbnb-instant-booking
Region       : ap-southeast-1
Role         : SageMakerExecutionRole-ITI113-Team14
Team ID      : team14
Student ID   : s1401
Task         : Binary Classification
Target       : instant_bookable
Data path    : Listings.csv
Snapshot     : 2021-03-01
numpy 1.26.4 | pandas 2.3.3


In [3]:
import pandas as pd

# Load just the first 5 rows to be fast
test_df = pd.read_csv(f"s3://{BUCKET}/{PREFIX99}/raw/master_recsys_dataset.csv", nrows=5)
print(test_df.columns.tolist())

['listing_id', 'name', 'host_id', 'host_since', 'host_location', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_total_listings_count', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'district', 'city', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bedrooms', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'instant_bookable']


In [4]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import uvicorn

# 1. Initialize the API
app = FastAPI(title="Airbnb ML Recommendation API")

# 2. Load and Pre-process the Data Globally
print("Loading master dataset into memory...")
try:
    # Ensure this file exists in the same directory as your notebook
    ##df = pd.read_csv("master_recsys_dataset.csv", low_memory=False)
    df = pd.read_csv(
    f"s3://{BUCKET}/{PREFIX99}/processed/master_recsys_dataset.csv", 
    encoding="utf-8", 
    encoding_errors="replace", 
    low_memory=False
)
    
    # Pre-calculate the scaled value score for fast API responses
    # A negative residual (pred_price - price) means the listing is underpriced (a good deal)
    df['deal_value'] = df['pred_price'] - df['price']
    
    scaler = MinMaxScaler()
    df['scaled_value_score'] = scaler.fit_transform(df[['deal_value']])
    print(f"✅ Dataset ready! Loaded {len(df)} listings.")
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    df = pd.DataFrame()

# 3. Define the Input Data Schema using Pydantic
class SearchQuery(BaseModel):
    city: str
    min_guests: int = 2
    max_price: float = 150.0
    w_value: float = 0.5
    w_convenience: float = 0.5

# 4. Create the Recommendation Endpoint
@app.post("/recommend")
def get_recommendations(query: SearchQuery):
    if df.empty:
        raise HTTPException(status_code=500, detail="Dataset not loaded. Check file path.")

    # Step A: Candidate Generation (Hard Filters)
    candidates = df[(df['city'].str.lower() == query.city.lower()) & 
                    (df['accommodates'] >= query.min_guests) &
                    (df['price'] <= query.max_price)].copy()
    
    if candidates.empty:
        return {"results": [], "message": f"No listings match your criteria in {query.city}."}

    # Step B: Apply the Machine Learning Re-ranking
    candidates['ml_score'] = (query.w_value * candidates['scaled_value_score']) + \
                             (query.w_convenience * candidates['instant_book_prob'])
                             
    # Step C: Sort and Return Top 10
    ranked = candidates.sort_values(by='ml_score', ascending=False).head(10)
    
    # Select only the relevant columns to send back to the user
    columns_to_return = ['listing_id', 'city', 'accommodates', 'price', 
                         'pred_price', 'instant_book_prob', 'ml_score']
                         
    # Convert the pandas dataframe into a list of dictionaries for the JSON response
    results = ranked[columns_to_return].to_dict(orient="records")
    
    return {"results": results, "count": len(results)}

# 5. Run the Server natively in Jupyter
if __name__ == "__main__":
    print("🚀 Starting FastAPI server on http://localhost:8000")
    print("⏳ The server is now listening for requests. (This cell will keep running)")
    
    # Configure the server
    config = uvicorn.Config(app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)
    
    # Run the server inside Jupyter's existing background event loop
    await server.serve()

Loading master dataset into memory...


✅ Dataset ready! Loaded 279724 listings.
🚀 Starting FastAPI server on http://localhost:8000
⏳ The server is now listening for requests. (This cell will keep running)


INFO:     Started server process [4229]


INFO:     Waiting for application startup.


INFO:     Application startup complete.


INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:49124 - "POST /recommend HTTP/1.1" 200 OK
